# State ISI and epoch durations (R3-18, R1-8)

Fig 3d plotters/bins unchanged. Active/quiet ISIs keep an interval only when **both** bounding saccades sit in the same raw behavior-state epoch (cross-epoch ISIs dropped). Epoch-duration histograms use unsmoothed annotations.


## 0. Setup


In [ ]:
%matplotlib inline
from __future__ import annotations

import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt

plt.ioff()

REPO = Path.cwd()
if not (REPO / "src" / "eye_tracking_system_tools").is_dir():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "src" / "eye_tracking_system_tools").is_dir():
            REPO = p
            break

sys.path.insert(0, str(REPO / "src"))
os.environ.setdefault("MPLCONFIGDIR", str(REPO / ".mplconfig"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(exist_ok=True)

from eye_tracking_system_tools.analysis.block_registry import load_registry
from eye_tracking_system_tools.analysis.event_cache import build_or_load_event_tables
from eye_tracking_system_tools.analysis.export_meta import load_params_yaml
from eye_tracking_system_tools.analysis.review_collect import review_answers_dir
from eye_tracking_system_tools.analysis.run_layout import resolve_figure_dirs

# Empty TAG overwrites outputs/review_answers_latest
TAG = ""
REGISTRY = REPO / "configs" / "paper_blocks.yaml"
PARAMS = REPO / "configs" / "analysis_params.yaml"
run = review_answers_dir(REPO / "outputs", TAG)
figures_dir, metadata_dir = resolve_figure_dirs(run)
print("REPO :", REPO)
print("run  :", run)
print("  figures :", figures_dir)
print("  metadata:", metadata_dir)

KEEP_TRACES = True


## 1. Load event tables


In [ ]:
params = load_params_yaml(PARAMS)
specs = load_registry(REGISTRY)
tables, cache_path, from_cache = build_or_load_event_tables(
    specs,
    params,
    metadata_dir,
    keep_traces=KEEP_TRACES,
    prefer_finalized=True,
)
print(f"blocks={len(tables.blocks)}  events={len(tables.all_saccades)}  cache={'hit' if from_cache else 'miss'}")
print(cache_path)


## 2. Build


In [ ]:
from eye_tracking_system_tools.analysis.figures_3d_isi import export_state_isi_and_epochs

written = export_state_isi_and_epochs(tables, run, show=True)
for name, path in written.items():
    print(name, "→", path)
